Modelo de Regressão Logística

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# importação dos arquivos
df_train = pd.read_csv("./dados_projeto/treino.csv")
df_test1 = pd.read_csv("./dados_projeto/teste1.csv")
df_test2 = pd.read_csv("./dados_projeto/teste2.csv")

In [ ]:
# tipos de variáveis
categorical_onehot_encoder_cols = ['AGRAVTABAC', 'AGRAVDROGA', 'AGRAVAIDS', 'AGRAVDIABE', 'HIV','POP_RUA', 'POP_LIBER', 'POP_IMIG', 'CS_SEXO', 'BACILOSC_E', 'RAIOX_TORA', 'CS_RACA', 'TRATAMENTO']
categorical_target_encoder_cols = ['SG_UF_NOT']
ordinal_cols = ['CS_ESCOL_N', 'CAT_IDADE']
numerical_cols = ['IDADE_ANOS']
target_col = ['ltfu']
feature_cols = categorical_onehot_encoder_cols + categorical_target_encoder_cols + ordinal_cols + numerical_cols

In [ ]:
# mapeia para maiúsculo as features em minúsculo
from sklearn.preprocessing import FunctionTransformer

def map_to_uppercase(df):
  if 'idade_anos' in df.columns:
    df['IDADE_ANOS'] = df['idade_anos']
  return df

age_formatter = FunctionTransformer(map_to_uppercase)

In [ ]:
# categoriza a idade da pessoa
def categorize_age(age):
  if age <= 29:
    return 'jovem_adulto'
  elif age <= 49:
    return 'adulto_meia_idade'
  elif age <= 64:
    return 'adulto_transicao_para_idoso'
  elif age >= 65:
    return 'idoso'
  else:
    return 'ignorado'

def apply_age_categorization(df):
  df['CAT_IDADE'] = df['IDADE_ANOS'].apply(categorize_age)
  return df

age_categorizer = FunctionTransformer(apply_age_categorization)

In [ ]:
# seleciona as colunas desejadas
def filter_cols(df):
  return df[feature_cols]

column_selector = FunctionTransformer(filter_cols)

In [ ]:
# mapeia os dados do dataframe
categorical_onehot_encoder_mapping = {
  'CS_RACA': {
    1: 'branca',
    2: 'preta',
    3: 'amarela',
    4: 'parda',
    5: 'indigena',
    9: 'ignorado',
    0: None
  },
  'AGRAVTABAC': {
    1: 'sim',
    2: 'nao',
    9: 'ignorado',
    0: None
  },
  'AGRAVDROGA': {
    1: 'sim',
    2: 'nao',
    9: 'ignorado',
    0: None
  },
  'AGRAVAIDS': {
    1: 'sim',
    2: 'nao',
    9: 'ignorado',
    0: None
  },
  'AGRAVDIABE': {
    1: 'sim',
    2: 'nao',
    9: 'ignorado',
    0: None
  },
  'HIV': {
    1: 'positivo',
    2: 'negativo',
    3: 'em_andamento',
    4: 'realizado',
    0: None
  },
  'POP_RUA': {
    1: 'sim',
    2: 'nao',
    0: None
  },
  'POP_LIBER': {
    1: 'sim',
    2: 'nao',
    0: None
  },
  'POP_IMIG': {
    1: 'sim',
    2: 'nao',
    0: None
  },
  'BACILOSC_E': {
    1: 'positiva',
    2: 'negativa',
    3: 'nao_realizada',
    4: 'nao_se_aplica',
    0: None
  },
  'RAIOX_TORA': {
    1: 'suspeito',
    2: 'normal',
    3: 'outra_patologia',
    4: 'nao_realizado',
    0: None
  },
  'TRATAMENTO': {
    1: 'caso_novo',
    2: 'recidiva',
    3: 'reingresso_apos_abandono',
    4: 'nao_sabe',
    5: 'transferencia',
    0: None
  }
}
categorical_target_encoder_mapping = {
  'SG_UF_NOT': {
    11: 'ro', 
    12: 'ac', 
    13: 'am', 
    14: 'rr', 
    15: 'pa', 
    16: 'ap', 
    17: 'to',
    21: 'ma', 
    22: 'pi', 
    23: 'ce', 
    24: 'rn', 
    25: 'pb', 
    26: 'pe', 
    27: 'al', 
    28: 'se', 
    29: 'ba',
    31: 'mg', 
    32: 'es', 
    33: 'rj', 
    35: 'sp',
    41: 'pr', 
    42: 'sc', 
    43: 'rs',
    50: 'ms', 
    51: 'mt', 
    52: 'go', 
    53: 'df'
  }
}
ordinal_mapping = {
  'CS_ESCOL_N': {
    0: 'analfabeto',
    1: '1a_a_4a_serie_incompleta_do_ef',
    2: '4a_serie_completa_do_ef',
    3: '5a_a_8a_serie_incompleta_do_ef',
    4: 'ensino_fundamental_completo',
    5: 'ensino_medio_incompleto',
    6: 'ensino_medio_completo',
    7: 'educacao_superior_incompleta',
    8: 'educacao_superior_completa',
    9: 'ignorado',
    10: 'nao_se_aplica'
  }
}

def map_df_data(df, cols_list, mapping_dict):
  for col in cols_list:
    if col in mapping_dict.keys():
      df[col] = df[col].map(mapping_dict[col])
  return df

def map_to_lowercase(df, col):
  df[col] = df[col].str.lower()
  return df

def apply_df_mapping(df):
  df_mapped = map_df_data(df, categorical_onehot_encoder_cols, categorical_onehot_encoder_mapping)
  df_mapped = map_df_data(df_mapped, categorical_target_encoder_cols, categorical_target_encoder_mapping)
  df_mapped = map_df_data(df_mapped, ordinal_cols, ordinal_mapping)
  df_mapped = map_to_lowercase(df_mapped, 'CS_SEXO')
  return df_mapped 

column_mapper = FunctionTransformer(apply_df_mapping)

In [ ]:
# separação das features e do alvo
X_train = df_train.drop(columns=target_col)
y_train = df_train[target_col[0]]

In [ ]:
# imputação dos missings
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, PowerTransformer

numerical_prep = Pipeline([
  ('imputer', IterativeImputer(random_state=32)),
  ('scaler', PowerTransformer())
])

categorical_onehot_encoder_prep = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent')),
  ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

categorical_target_encoder_prep = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent')),
  ('encoder', TargetEncoder(cv=5))
])

ordinal_prep = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent')),
  ('encoder', TargetEncoder(cv=5))
])

preprocessor = ColumnTransformer([
  ('num', numerical_prep, numerical_cols),
  ('cat_onehot_encoder', categorical_onehot_encoder_prep, categorical_onehot_encoder_cols),
  ('cat_target_encoder', categorical_target_encoder_prep, categorical_target_encoder_cols),
  ('ord', ordinal_prep, ordinal_cols)
])

In [ ]:
# logistic regression
from sklearn.linear_model import LogisticRegression

logreg_model = LogisticRegression(max_iter=1000, penalty='elasticnet', solver='saga', n_jobs=-1)

In [ ]:
# pipeline
pipeline = Pipeline(
  steps = [
    ('age_format', age_formatter),
    ('age_cat', age_categorizer),
    ('selector', column_selector),
    ('mapper', column_mapper),
    ('pre', preprocessor),
    ('clf', logreg_model)
  ]
)

In [ ]:
# grid
grid = {
  'clf__C':np.logspace(-3, 2, 100),
  'clf__l1_ratio':np.linspace(0, 1, 100),
  'clf__class_weight':['balanced', None]
}

In [ ]:
# randomized search
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
  estimator=pipeline,
  param_distributions=grid,
  n_iter=50,
  cv=3,
  random_state=32,
  n_jobs=-1,
  verbose=3,
  scoring=['roc_auc', 'accuracy', 'recall', 'precision', 'f1'],
  refit='f1'
)

In [ ]:
# treina o modelo
random_search.fit(X_train, y_train)

In [ ]:
metrics = ["rank_test_recall", "rank_test_precision", "rank_test_f1",  "rank_test_roc_auc", "mean_fit_time", "mean_score_time", "mean_test_recall", "mean_test_precision", "mean_test_f1", "mean_test_roc_auc"]

# melhores hiperparâmetros
def show_best_hyperparams():
  return (f"melhores hiperparâmetros: {random_search.best_params_}")

# métricas treino
def show_metrics():
  return pd.DataFrame(random_search.cv_results_)[metrics].sort_values(by=["rank_test_f1"]).head()

In [ ]:
# mostra melhores hiperparâmetros e métricas do treino inicial
print(show_best_hyperparams())
show_metrics()

In [ ]:
# testa o modelo no df_test1
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, RocCurveDisplay

X_test1 = df_test1.drop(columns=target_col)
y_test1 = df_test1[target_col[0]]

y_pred_test1 = random_search.predict(X_test1)
y_proba_test1 = random_search.predict_proba(X_test1)[:, 1]

def calc_scores(y_test, y_pred, y_proba):
  scores = {
    "accuracy":[accuracy_score(y_test, y_pred)],
    "recall":[recall_score(y_test, y_pred)],
    "precision":[precision_score(y_test, y_pred)],
    "f1_score":[f1_score(y_test, y_pred)],
    "auc_roc":[roc_auc_score(y_test, y_proba)]
    }
  return scores

scores_test1 = calc_scores(y_test1, y_pred_test1, y_proba_test1)
pd.DataFrame(scores_test1, index=["Teste1"])

In [ ]:
# curva ROC test1
def show_graphic_auc_roc(y_test, y_proba):
  RocCurveDisplay.from_predictions(y_test, y_proba, plot_chance_level=True)

show_graphic_auc_roc(y_test1, y_proba_test1)

In [ ]:
# retreina o modelo
df_combined = pd.concat([df_train, df_test1], ignore_index=True)

X_combined = df_combined.drop(columns=target_col)
y_combined = df_combined[target_col[0]]

random_search.fit(X_combined, y_combined)

In [ ]:
# mostra melhores hiperparâmetros e métricas do treino final
print(show_best_hyperparams())
show_metrics()

In [ ]:
# testa o modelo no df_test2
X_test2 = df_test2.drop(columns=target_col)
y_test2 = df_test2[target_col[0]]

y_pred_test2 = random_search.predict(X_test2)
y_proba_test2 = random_search.predict_proba(X_test2)[:, 1]

scores_test2 = calc_scores(y_test2, y_pred_test2, y_proba_test2)
pd.DataFrame(scores_test2, index=["Teste2"])

In [ ]:
# curva ROC test2
show_graphic_auc_roc(y_test2, y_proba_test2)

In [ ]:
# feature importances
feature_names = random_search.best_estimator_['pre'].get_feature_names_out()
coefficients = random_search.best_estimator_['clf'].coef_[0]

feature_importance_df = pd.DataFrame({'feature':feature_names, 'coef':coefficients})
feature_importance_df['abs_coef'] = np.abs(feature_importance_df['coef'])
feature_importance_df = feature_importance_df.sort_values(by='abs_coef', ascending=False)

top_features = feature_importance_df.head(30)

plt.figure(figsize=(10, 12))
plt.barh(top_features['feature'][::-1], top_features['coef'][::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top Features')
plt.tight_layout()
plt.show()

In [ ]:
# extrai o melhor modelo
import cloudpickle

open('logreg_model.pkl', 'wb').write(cloudpickle.dumps(random_search.best_estimator_))